In [1]:
!pip install \
torch \
torchvision \
streamlit \
scikit-learn \
pillow \
opencv-python \
numpy \
matplotlib \
wget


Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os

# Tentukan direktori tempat dataset Caltech 101 disimpan
dataset_dir = 'C:/Users/Fitra/Documents/RSIP/101_ObjectCategories'

# Cek apakah direktori dataset ada
if not os.path.exists(dataset_dir):
    raise FileNotFoundError(f"Dataset tidak ditemukan di {dataset_dir}. Pastikan path sudah benar.")

# Ambil nama-nama kelas dari folder dataset
class_names = os.listdir(dataset_dir)

# Tampilkan nama kelas yang ada di dataset
print("Kelas yang tersedia dalam dataset:")
for class_name in class_names:
    print(f"- {class_name}")

print("Dataset siap digunakan.")


Kelas yang tersedia dalam dataset:
- accordion
- airplanes
- anchor
- ant
- BACKGROUND_Google
- barrel
- bass
- beaver
- binocular
- bonsai
- brain
- brontosaurus
- buddha
- butterfly
- camera
- cannon
- car_side
- ceiling_fan
- cellphone
- chair
- chandelier
- cougar_body
- cougar_face
- crab
- crayfish
- crocodile
- crocodile_head
- cup
- dalmatian
- dollar_bill
- dolphin
- dragonfly
- electric_guitar
- elephant
- emu
- euphonium
- ewer
- Faces
- Faces_easy
- ferry
- flamingo
- flamingo_head
- garfield
- gerenuk
- gramophone
- grand_piano
- hawksbill
- headphone
- hedgehog
- helicopter
- ibis
- inline_skate
- joshua_tree
- kangaroo
- ketch
- lamp
- laptop
- Leopards
- llama
- lobster
- lotus
- mandolin
- mayfly
- menorah
- metronome
- minaret
- Motorbikes
- nautilus
- octopus
- okapi
- pagoda
- panda
- pigeon
- pizza
- platypus
- pyramid
- revolver
- rhino
- rooster
- saxophone
- schooner
- scissors
- scorpion
- sea_horse
- snoopy
- soccer_ball
- stapler
- starfish
- stegosaurus
- st

In [4]:
import torch
from torchvision import models, transforms
from PIL import Image

# Memuat model ResNet-50 pretrained
model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])  # Hapus lapisan terakhir
model = model.eval()  # Mengatur model ke mode evaluasi

# Transformasi gambar untuk ResNet-50
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Fungsi untuk mengekstrak fitur dari gambar
def extract_features(image_path):
    image = Image.open(image_path).convert('RGB')  # Pastikan gambar dalam format RGB
    image_tensor = transform(image).unsqueeze(0)  # Mengubah gambar menjadi tensor dan menambahkan dimensi batch
    
    with torch.no_grad():
        features = model(image_tensor)  # Mendapatkan fitur dari model
    return features.flatten()  # Meratakan fitur ke bentuk vektor


C:\Users\Fitra\AppData\Roaming\Python\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Fitra\AppData\Roaming\Python\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
import numpy as np

# Menyimpan fitur dan label
features_list = []
labels_list = []

# Ambil nama kelas dari folder di dataset
class_names = os.listdir(dataset_dir)

# Ekstraksi fitur untuk setiap gambar di dataset
for class_idx, class_name in enumerate(class_names):
    class_folder = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_folder):
        for image_name in os.listdir(class_folder):
            image_path = os.path.join(class_folder, image_name)
            try:
                features = extract_features(image_path)
                features_list.append(features.numpy())
                labels_list.append(class_idx)
            except Exception as e:
                print(f"Error memproses gambar: {image_path} - {e}")

# Menyimpan fitur dan label sebagai array numpy
features_matrix = np.array(features_list)
labels_matrix = np.array(labels_list)

print("Ekstraksi fitur selesai.")


Ekstraksi fitur selesai.


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# Fungsi untuk mencari gambar yang mirip
def find_similar_images(query_image_path, top_k=5):
    # Ekstraksi fitur gambar query
    query_features = extract_features(query_image_path).reshape(1, -1)
    
    # Hitung cosine similarity dengan semua gambar di dataset
    similarities = cosine_similarity(query_features, features_matrix)
    
    # Ambil indeks gambar yang paling mirip
    similar_indices = np.argsort(similarities[0])[-top_k:][::-1]  # Top-k tertinggi
    similar_images = [(class_names[labels_matrix[idx]], similarities[0][idx]) for idx in similar_indices]
    
    return similar_images


In [7]:
import random
import os

# Tentukan direktori dataset
dataset_dir = 'C:/Users/Fitra/Documents/RSIP/101_ObjectCategories'

# Pilih kelas secara acak
class_names = os.listdir(dataset_dir)
random_class = random.choice(class_names)

# Pilih gambar secara acak dari kelas yang dipilih
class_dir = os.path.join(dataset_dir, random_class)
image_names = os.listdir(class_dir)
random_image_name = random.choice(image_names)
query_image_path = os.path.join(class_dir, random_image_name)

# Cari gambar yang paling mirip
top_k = 5
similar_images = find_similar_images(query_image_path, top_k)

# Tampilkan hasil pencarian
print(f"Gambar query: {query_image_path}")
print("Top 5 gambar yang mirip:")
for idx, (class_name, similarity_score) in enumerate(similar_images):
    print(f"Rank {idx+1}:")
    print(f"   - Kelas: {class_name}")
    print(f"   - Similarity Score: {similarity_score:.4f}")


Gambar query: C:/Users/Fitra/Documents/RSIP/101_ObjectCategories\cup\image_0025.jpg
Top 5 gambar yang mirip:
Rank 1:
   - Kelas: cup
   - Similarity Score: 1.0000
Rank 2:
   - Kelas: cup
   - Similarity Score: 0.8880
Rank 3:
   - Kelas: cup
   - Similarity Score: 0.8673
Rank 4:
   - Kelas: cup
   - Similarity Score: 0.8666
Rank 5:
   - Kelas: cup
   - Similarity Score: 0.8569


In [8]:
import streamlit as st
from PIL import Image

# Streamlit UI untuk pencarian gambar
st.title("Reverse Image Search Engine")

# Mengunggah gambar untuk pencarian
uploaded_file = st.file_uploader("Upload Image", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    # Menampilkan gambar yang diunggah
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded Image", use_column_width=True)
    
    # Simpan gambar sementara untuk diproses
    query_image_path = './temp_image.jpg'
    image.save(query_image_path)

    # Menemukan gambar yang mirip
    top_k = 5
    similar_images = find_similar_images(query_image_path, top_k)

    # Menampilkan gambar yang mirip
    st.write("Top 5 similar images:")
    for idx, (class_name, similarity_score) in enumerate(similar_images):
        st.write(f"Rank {idx+1}:")
        st.write(f"   - Kelas: {class_name}")
        st.write(f"   - Similarity Score: {similarity_score:.4f}")


2024-12-03 10:25:34.340 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-03 10:25:34.675 
  command:

    streamlit run C:\Users\Fitra\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2024-12-03 10:25:34.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-03 10:25:34.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-03 10:25:34.677 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-03 10:25:34.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-03 10:25:34.679 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2024-12-03 10:25:34.679 Thre